# Local thermal average power around insect observations

Compute average DCT power in the `<1.25`, `1.25–2.5`, and `>2.5` cycles/m bands for a point-centred window around every matched 2022 insect observation and export the results.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from rasterio.transform import rowcol
from rasterio.windows import Window, bounds as window_bounds
from shapely.geometry import box
from tqdm.auto import tqdm

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from get_1d_psd import get_1d_psd
from thermal_image_helpers import load_shp, load_thermal

output_root = project_root / "data" / "derived" / "insect_analysis"
output_root.mkdir(parents=True, exist_ok=True)


## Load and standardize insect observations

In [2]:
pol_path = project_root.joinpath(
    "./data/raw/insect_data/data_observaties_pollinatoren3.xlsx"
)

cols = [
    "Datum",
    "Periode",
    "Locatie",
    "Site",
    "Transect",
    "Soortengroep",
    "Aantal",
    "Genus",
    "Lat",
    "Lon",
    "Nauwkeurigheid",
    "Meettoestel",
    "Jaar",
]

## Data 2020
df20 = pd.read_excel(pol_path, sheet_name="Data2020 + genus")
df20["Lat"] = df20.apply(
    lambda x: x["Lat (GPS)"] if x["Lat (GPS)"] != "-" else x["Lat (GSM)"], axis=1
)
df20["Lon"] = df20.apply(
    lambda x: x["Lon (GPS)"] if x["Lon (GPS)"] != "-" else x["Lon (GSM)"], axis=1
)
# Accuracy of GPS (not smartphone) is advertised to be 3m
df20["Nauwkeurigheid"] = df20.apply(
    lambda x: 3 if x["Lon (GPS)"] != "-" else x["Nauwk"], axis=1
)
df20["Meettoestel"] = df20.apply(
    lambda x: "GPS" if x["Lon (GPS)"] != "-" else "GSM", axis=1
)
df20.rename(columns={"Soortgroep": "Soortengroep", "periode": "Periode"}, inplace=True)
df20["Jaar"] = 2020

## Data 2021
df21 = pd.read_excel(pol_path, sheet_name="DATA2021 + genus")

df21.Soortengroep = df21.Soortengroep.map(
    lambda x: {"Bee": "Bijen", "Butterfly": "Dagvlinders"}[x]
)
# TODO: Missing accuracy data here set to 10 temporary
df21["Nauwkeurigheid"] = 10  # df21.apply(lambda x: )
df21.rename(columns={"Beheer": "Site"}, inplace=True)
df21["Meettoestel"] = "GSM"
df21["Jaar"] = 2021

## Data 2022
pol_path = project_root.joinpath(
    "./data/raw/insect_data/data_observaties_pollinatoren2.xlsx"
)
df22 = pd.read_excel(pol_path)
df22.rename(columns={"periode": "Periode"}, inplace=True)
df22["Meettoestel"] = "GSM"
df22.Soortengroep = df22.Soortengroep.map(
    lambda x: {"Bee": "Bijen", "Butterfly": "Dagvlinders"}[x]
)
df22["Jaar"] = 2022

# Concat data
df = pd.concat([df20[cols], df21[cols], df22[cols]], ignore_index=True)
df.Transect = df.Transect.str.strip()  # Stripping spaces

# Convert to GDF
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Lon, df.Lat))
gdf.Datum = pd.to_datetime(gdf.Datum)

gdf = gdf.set_crs("WGS84")
gdf = gdf.to_crs("EPSG:31370")
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y
#gdf.drop(columns=["Lat", "Lon"], inplace=True)
gdf.Site = gdf["Site"].map(lambda x: {"B": "blok", "S": "sinus"}[x])
gdf.Locatie = gdf["Locatie"].map(
    lambda x: {
        "P": "Palingbeek",
        "M": "Muziekbos",
        "W1": "Waarmaarde",
        "W2": "Waarmaarde2",
        "K": "Kemmelberg",
        "E": "Westbekesluis",
    }[x]
)

# Compensating for Aantal by duplicating rows as multiple observations
for i in gdf.Aantal.unique():
    try:
        i = int(i)
    except:
        continue

    if i > 1:
        gdf = pd.concat([gdf] + [gdf[gdf.Aantal == i]] * (i - 1))

gdf["Aantal"] = 1

gdf = gdf.reset_index()


In [3]:
FLIGHTS = [
    {
        "field": "Waarmaarde",
        "date": "20220428",
        "sinus_tif": "./data/local/Waarmaarde/20220428/sinus/thermal/ortho/220814_Waarm_Thermal_Georef.tif",
        "blok_tif": "./data/local/Waarmaarde/20220428/blok/thermal/ortho/220814_Waarm_Thermal_Georef.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_WaarmaardeOost_sinus_maaien_2021_09.shp",
        "blok_shp": "./data/raw/shape_files/WaarmaardeOost_blok_maaien_2021_09.shp",
    },
    {
        "field": "Muziekbos",
        "date": "20220617",
        "sinus_tif": "./data/local/Muziekbos/20220617/sinus/thermal/ortho/220617_Muziek_fl2_Thermal.tif",
        "blok_tif": "./data/local/Muziekbos/20220617/blok/thermal/ortho/220617_Muziek_fl1_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_Muziekbos_sinus_maaien_2021_09.shp",
        "blok_shp": "./data/raw/shape_files/corrected_Muziekbos_blok_maaien_2021_09.shp",
    },
    {
        "field": "Palingbeek",
        "date": "20220617",
        "sinus_tif": "./data/local/Palingbeek/20220617/sinus/thermal/ortho/20220617_Paling_Thermal.tif",
        "blok_tif": "./data/local/Palingbeek/20220617/blok/thermal/ortho/20220617_Paling_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_Palingbeek_sinus_maaien_2021_09.shp",
        "blok_shp": "./data/raw/shape_files/Palingbeek_blok_maaien_2021_09.shp",
    },
    {
        "field": "Muziekbos",
        "date": "20220705",
        "sinus_tif": "./data/local/Muziekbos/20220705/sinus/thermal/ortho/220705_Muziek_Sin_Thermal.tif",
        "blok_tif": "./data/local/Muziekbos/20220705/blok/thermal/ortho/220705_Muziek_Block_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_Muziekbos_sinus_maaien_2022_06.shp",
        "blok_shp": "./data/raw/shape_files/corrected_Muziekbos_blok_maaien_2022_06.shp",
    },
    {
        "field": "Waarmaarde",
        "date": "20220705",
        "sinus_tif": "./data/local/Waarmaarde/20220705/sinus/thermal/ortho/220507_Waarm_Thermal.tif",
        "blok_tif": "./data/local/Waarmaarde/20220705/blok/thermal/ortho/220507_Waarm_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_WaarmaardeOost_sinus_maaien_2022_06.shp",
        "blok_shp": "./data/raw/shape_files/corrected_WaarmaardeOost_blok_maaien_2022_06.shp",
    },
    {
        "field": "Muziekbos",
        "date": "20220831",
        "sinus_tif": "./data/local/Muziekbos/20220831/sinus/thermal/ortho/220831_Muziek_Thermal.tif",
        "blok_tif": "./data/local/Muziekbos/20220831/blok/thermal/ortho/220831_Muziek_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_Muziekbos_sinus_maaien_2022_06.shp",
        "blok_shp": "./data/raw/shape_files/corrected_Muziekbos_blok_maaien_2022_06.shp",
    },
    {
        "field": "Waarmaarde",
        "date": "20220831",
        "sinus_tif": "./data/local/Waarmaarde/20220831/sinus/thermal/ortho/220831_Waarm_Thermal.tif",
        "blok_tif": "./data/local/Waarmaarde/20220831/blok/thermal/ortho/220831_Waarm_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_WaarmaardeOost_sinus_maaien_2022_06.shp",
        "blok_shp": "./data/raw/shape_files/corrected_WaarmaardeOost_blok_maaien_2022_06.shp",
    },
    {
        "field": "Palingbeek",
        "date": "20220901",
        "sinus_tif": "./data/local/Palingbeek/20220901/sinus/thermal/ortho/220901_Paling_Thermal.tif",
        "blok_tif": "./data/local/Palingbeek/20220901/blok/thermal/ortho/220901_Paling_Thermal.tif",
        "sinus_shp": "./data/raw/shape_files/corrected_Palingbeek_sinus_maaien_2022_06.shp",
        "blok_shp": "./data/raw/shape_files/corrected_Palingbeek_blok_maaien_2022_06.shp",
    },
]

pd.DataFrame(FLIGHTS)


,field,date,sinus_tif,blok_tif,sinus_shp,blok_shp
0,Waarmaarde,20220428,./data/local/Waarmaarde/20220428/sinus/thermal...,./data/local/Waarmaarde/20220428/blok/thermal/...,./data/raw/shape_files/corrected_WaarmaardeOos...,./data/raw/shape_files/WaarmaardeOost_blok_maa...
1,Muziekbos,20220617,./data/local/Muziekbos/20220617/sinus/thermal/...,./data/local/Muziekbos/20220617/blok/thermal/o...,./data/raw/shape_files/corrected_Muziekbos_sin...,./data/raw/shape_files/corrected_Muziekbos_blo...
2,Palingbeek,20220617,./data/local/Palingbeek/20220617/sinus/thermal...,./data/local/Palingbeek/20220617/blok/thermal/...,./data/raw/shape_files/corrected_Palingbeek_si...,./data/raw/shape_files/Palingbeek_blok_maaien_...
3,Muziekbos,20220705,./data/local/Muziekbos/20220705/sinus/thermal/...,./data/local/Muziekbos/20220705/blok/thermal/o...,./data/raw/shape_files/corrected_Muziekbos_sin...,./data/raw/shape_files/corrected_Muziekbos_blo...
4,Waarmaarde,20220705,./data/local/Waarmaarde/20220705/sinus/thermal...,./data/local/Waarmaarde/20220705/blok/thermal/...,./data/raw/shape_files/corrected_WaarmaardeOos...,./data/raw/shape_files/corrected_WaarmaardeOos...
5,Muziekbos,20220831,./data/local/Muziekbos/20220831/sinus/thermal/...,./data/local/Muziekbos/20220831/blok/thermal/o...,./data/raw/shape_files/corrected_Muziekbos_sin...,./data/raw/shape_files/corrected_Muziekbos_blo...
6,Waarmaarde,20220831,./data/local/Waarmaarde/20220831/sinus/thermal...,./data/local/Waarmaarde/20220831/blok/thermal/...,./data/raw/shape_files/corrected_WaarmaardeOos...,./data/raw/shape_files/corrected_WaarmaardeOos...
7,Palingbeek,20220901,./data/local/Palingbeek/20220901/sinus/thermal...,./data/local/Palingbeek/20220901/blok/thermal/...,./data/raw/shape_files/corrected_Palingbeek_si...,./data/raw/shape_files/corrected_Palingbeek_bl...


## Match thermal flights to insect sampling periods

In [4]:
thermal_insect_period_mapping = {
    ("Waarmaarde", "20220428"): [
        "p1",
        "p2",
        "p3",
    ],  # p1: 2022-04-20, p2: 2022-05-18, p3: 2022-06-14
    ("Waarmaarde", "20220705"): [],
    ("Waarmaarde", "20220831"): [
        "p4",
        "p5",
        "p6",
    ],  # p4: 2022-08-02, p5: 2022-08-25, p6: 2022-09-30
    ("Muziekbos", "20220617"): [
        "p1",
        "p2",
        "p3",
    ],  # p1: 2022-04-15, p2: 2022-05-14, p3: 2022-06-20
    ("Muziekbos", "20220705"): [],
    ("Muziekbos", "20220831"): [
        "p4",
        "p5",
        "p6",
    ],  # p4: 2022-08-05, p5: 2022-08-31, p6: 2022-09-21
    ("Palingbeek", "20220617"): [
        "p1",
        "p2",
        "p3",
    ],  # p1: 2022-04-12, p2: 2022-05-11, p3: 2022-06-21
    ("Palingbeek", "20220901"): [
        "p4",
        "p5",
        "p6",
    ],  # p4: 2022-08-05, p5: 2022-08-24, p6:2022-09-22
}

## Compute local thermal average power

In [5]:
window_size_m = 2
window_size_m = 4
window_size_m = 8
gsd_m = 0.064
window_size_pixels = round(window_size_m / gsd_m)
real_window_size_m = window_size_pixels * gsd_m
power_columns = [
    "average_power_lt_1p25_cpm",
    "average_power_1p25_2p5_cpm",
    "average_power_gt_2p5_cpm",
]

igdf = gdf.copy()
igdf = igdf[igdf.Datum >= datetime(year=2022, month=1, day=1)].copy()
igdf["insect_id"] = np.arange(len(igdf))
igdf[power_columns] = np.nan
igdf["part"] = ""
igdf["thermal_source"] = ""

thermal_shape_manifest = [
    (flight_info, management)
    for flight_info in FLIGHTS
    for management in ("sinus", "blok")
]
for flight_info, management in tqdm(thermal_shape_manifest):
    site = flight_info["field"]
    date_string = flight_info["date"]
    therm_path = project_root / flight_info[f"{management}_tif"]
    shape_path = project_root / flight_info[f"{management}_shp"]
    print(site, date_string, management)

    shp = load_shp(shape_path)
    mow_shape = shp.geometry.iloc[1]
    no_mow_shape = shp.geometry.iloc[0]
    for g in shp.geometry.iloc[2:]:
        no_mow_shape = no_mow_shape.union(g)
    outer_shape = mow_shape.union(no_mow_shape)

    # Get all insects that are relevant for this thermal images, meaning:
    # All insects that were recorded AFTER the thermal flight took place and BEFORE the next flight.
    relevant_periods = thermal_insect_period_mapping[(site, date_string)]
    selected_insects = igdf[
        (igdf.Locatie == site)
        & (igdf.Site == management)
        & (igdf.Datum >= datetime(2022, 1, 1))
        & (igdf.Periode.isin(relevant_periods))
    ]

    print(len(selected_insects))

    selected_insects = selected_insects[(selected_insects.geometry.within(outer_shape))]
    print("Lenght after shape intersection: ", len(selected_insects))
    igdf.loc[selected_insects.index, "thermal_source"] = therm_path.stem
    igdf.loc[selected_insects.index, "mow_shape_key"] = shape_path.stem

    if len(selected_insects) == 0:
        continue

    # Load with some buffer to also get thermal data outside of parcel
    thermal = load_thermal(
        therm_path, outer_shape.buffer(window_size_m / 2 + 0.5), gsd_m=gsd_m
    )

    def process_part(row):
        point = row["geometry"]
        # Check if insect is in mown area or not
        part = "mow" if point.within(mow_shape) else "no_mow"
        return part

    def process_therm(row):
        point = row["geometry"]
        center_row, center_col = rowcol(thermal.rio.transform(), point.x, point.y)
        # For even widths, the extra pixel lies on the positive row/column side.
        offset = (window_size_pixels - 1) // 2
        row_start, col_start = center_row - offset, center_col - offset
        row_stop = row_start + window_size_pixels
        col_stop = col_start + window_size_pixels
        local_thermal = thermal.data[row_start:row_stop, col_start:col_stop]

        if (
            row_start < 0
            or col_start < 0
            or local_thermal.shape != (window_size_pixels, window_size_pixels)
            or not np.isfinite(local_thermal).all()
        ):
            return (np.nan, np.nan, np.nan)

        psd, freqs = get_1d_psd(
            local_thermal - local_thermal.mean(),
            transform_type="DCT",
            bin_size=1,
            average_power_band=True,
        )
        freqs_cpm = freqs[:-1] / real_window_size_m
        band_masks = (
            (freqs_cpm > 0) & (freqs_cpm < 1.25),
            (freqs_cpm >= 1.25) & (freqs_cpm <= 2.5),
            freqs_cpm > 2.5,
        )
        return tuple(psd[mask].mean() if mask.any() else np.nan for mask in band_masks)

    powers = selected_insects.apply(process_therm, axis=1, result_type="expand")
    powers.columns = power_columns
    igdf.loc[selected_insects.index, power_columns] = powers
    igdf.loc[selected_insects.index, "part"] = selected_insects.apply(
        process_part, axis=1
    )
igdf

  0%|          | 0/16 [00:00<?, ?it/s]

Waarmaarde 20220428 sinus
153
Lenght after shape intersection:  113
Waarmaarde 20220428 blok
126
Lenght after shape intersection:  114
Muziekbos 20220617 sinus
241
Lenght after shape intersection:  147
Muziekbos 20220617 blok
176
Lenght after shape intersection:  160
Palingbeek 20220617 sinus
234
Lenght after shape intersection:  195
Palingbeek 20220617 blok
175
Lenght after shape intersection:  163
Muziekbos 20220705 sinus
0
Lenght after shape intersection:  0
Muziekbos 20220705 blok
0
Lenght after shape intersection:  0
Waarmaarde 20220705 sinus
0
Lenght after shape intersection:  0
Waarmaarde 20220705 blok
0
Lenght after shape intersection:  0
Muziekbos 20220831 sinus
266
Lenght after shape intersection:  199
Muziekbos 20220831 blok
194
Lenght after shape intersection:  170
Waarmaarde 20220831 sinus
144
Lenght after shape intersection:  88
Waarmaarde 20220831 blok
94
Lenght after shape intersection:  90
Palingbeek 20220901 sinus
320
Lenght after shape intersection:  272
Palingbeek 2

,index,Datum,Periode,Locatie,Site,Transect,Soortengroep,Aantal,Genus,Lat,...,geometry,x,y,insect_id,average_power_lt_1p25_cpm,average_power_1p25_2p5_cpm,average_power_gt_2p5_cpm,part,thermal_source,mow_shape_key
6537,6537,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822648,...,POINT (47367.572 169029.809),47367.571886,169029.809260,0,0.010410,0.000111,0.000003,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6538,6538,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822830,...,POINT (47302.477 169051.314),47302.477335,169051.313956,1,0.024952,0.000320,0.000007,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6539,6539,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.823142,...,POINT (47358.185 169084.918),47358.185214,169084.917783,2,0.008192,0.000144,0.000005,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6540,6540,2022-04-12,p1,Palingbeek,blok,T1,Dagvlinders,1,Aglais,50.822702,...,POINT (47402.103 169035.133),47402.103305,169035.133339,3,0.014603,0.000210,0.000007,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6541,6541,2022-04-12,p1,Palingbeek,blok,T2,Dagvlinders,1,Aglais,50.823071,...,POINT (47375.42 169076.69),47375.420334,169076.690131,4,0.041377,0.000182,0.000007,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11542,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,POINT (99681.29 162242.761),99681.290204,162242.760686,4251,0.023147,0.001418,0.000094,mow,220831_Muziek_Thermal,corrected_Muziekbos_sinus_maaien_2022_06
11554,7211,2022-06-18,p3,Kemmelberg,sinus,T2,Bijen,1,Bombus,50.773146,...,POINT (39183.618 163688.878),39183.618465,163688.877504,4252,NaN,NaN,NaN,,,NaN
11555,9265,2022-06-18,p3,Kemmelberg,sinus,V,Dagvlinders,1,Maniola,50.772965,...,POINT (39162.54 163669.184),39162.539726,163669.183964,4253,NaN,NaN,NaN,,,NaN
11556,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,POINT (99728.434 162363.835),99728.434030,162363.835182,4254,0.053411,0.000939,0.000040,no_mow,220617_Muziek_fl1_Thermal,corrected_Muziekbos_blok_maaien_2021_09


In [27]:
igdf.columns

Index(['index', 'Datum', 'Periode', 'Locatie', 'Site', 'Transect',
       'Soortengroep', 'Aantal', 'Genus', 'Lat', 'Lon', 'Nauwkeurigheid',
       'Meettoestel', 'Jaar', 'geometry', 'x', 'y', 'insect_id',
       'average_power_lt_1p25_cpm', 'average_power_1p25_2p5_cpm',
       'average_power_gt_2p5_cpm', 'part', 'thermal_source', 'mow_shape_key'],
      dtype='str')

In [28]:
igdf

,index,Datum,Periode,Locatie,Site,Transect,Soortengroep,Aantal,Genus,Lat,...,geometry,x,y,insect_id,average_power_lt_1p25_cpm,average_power_1p25_2p5_cpm,average_power_gt_2p5_cpm,part,thermal_source,mow_shape_key
6537,6537,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822648,...,POINT (47367.572 169029.809),47367.571886,169029.809260,0,0.010410,0.000111,0.000003,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6538,6538,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822830,...,POINT (47302.477 169051.314),47302.477335,169051.313956,1,0.024952,0.000320,0.000007,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6539,6539,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.823142,...,POINT (47358.185 169084.918),47358.185214,169084.917783,2,0.008192,0.000144,0.000005,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6540,6540,2022-04-12,p1,Palingbeek,blok,T1,Dagvlinders,1,Aglais,50.822702,...,POINT (47402.103 169035.133),47402.103305,169035.133339,3,0.014603,0.000210,0.000007,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6541,6541,2022-04-12,p1,Palingbeek,blok,T2,Dagvlinders,1,Aglais,50.823071,...,POINT (47375.42 169076.69),47375.420334,169076.690131,4,0.041377,0.000182,0.000007,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11542,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,POINT (99681.29 162242.761),99681.290204,162242.760686,4251,0.023147,0.001418,0.000094,mow,220831_Muziek_Thermal,corrected_Muziekbos_sinus_maaien_2022_06
11554,7211,2022-06-18,p3,Kemmelberg,sinus,T2,Bijen,1,Bombus,50.773146,...,POINT (39183.618 163688.878),39183.618465,163688.877504,4252,NaN,NaN,NaN,,,NaN
11555,9265,2022-06-18,p3,Kemmelberg,sinus,V,Dagvlinders,1,Maniola,50.772965,...,POINT (39162.54 163669.184),39162.539726,163669.183964,4253,NaN,NaN,NaN,,,NaN
11556,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,POINT (99728.434 162363.835),99728.434030,162363.835182,4254,0.053411,0.000939,0.000040,no_mow,220617_Muziek_fl1_Thermal,corrected_Muziekbos_blok_maaien_2021_09


In [29]:
igdf.Nauwkeurigheid.median()

np.float64(4.0)

## Validate and export

In [23]:
matched_insects = igdf[igdf.thermal_source != ""]
assert len(matched_insects) == 1906, f"Expected 1906 matched insects, found {len(matched_insects)}"
power_values = matched_insects[power_columns].to_numpy()
complete_triplets = np.isfinite(power_values).all(axis=1) | np.isnan(power_values).all(axis=1)
assert complete_triplets.all(), "Power bands must be either all finite or all NaN"
pd.DataFrame(
    {
        "matched_insects": [len(matched_insects)],
        "missing_average_power": [np.isnan(power_values).all(axis=1).sum()],
        "thermal_sources": [matched_insects.thermal_source.nunique()],
    }
)


,matched_insects,missing_average_power,thermal_sources
0,1906,1,7


In [24]:
output_path = output_root / (
    f"insect_local_thermal_variation_v2_{window_size_m:g}x{window_size_m:g}m_"
    f"{gsd_m:g}m_gsd_1p25-2p5_cycles_per_m.xlsx"
)
matched_insects.to_excel(output_path, index=False)
exported = pd.read_excel(output_path)
assert len(exported) == len(matched_insects)
assert set(power_columns).issubset(exported.columns)
assert {"x", "y"}.issubset(exported.columns)
#assert {"Lat", "Lon"}.isdisjoint(exported.columns)

In [18]:
matched_insects

,index,Datum,Periode,Locatie,Site,Transect,Soortengroep,Aantal,Genus,Lat,...,geometry,x,y,insect_id,average_power_lt_1p25_cpm,average_power_1p25_2p5_cpm,average_power_gt_2p5_cpm,part,thermal_source,mow_shape_key
6537,6537,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822648,...,POINT (47367.572 169029.809),47367.571886,169029.809260,0,0.016384,0.000366,0.000014,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6538,6538,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822830,...,POINT (47302.477 169051.314),47302.477335,169051.313956,1,0.043548,0.001478,0.000032,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6539,6539,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.823142,...,POINT (47358.185 169084.918),47358.185214,169084.917783,2,0.018005,0.000431,0.000021,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6540,6540,2022-04-12,p1,Palingbeek,blok,T1,Dagvlinders,1,Aglais,50.822702,...,POINT (47402.103 169035.133),47402.103305,169035.133339,3,0.042651,0.001199,0.000029,no_mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
6541,6541,2022-04-12,p1,Palingbeek,blok,T2,Dagvlinders,1,Aglais,50.823071,...,POINT (47375.42 169076.69),47375.420334,169076.690131,4,0.037088,0.000728,0.000031,mow,20220617_Paling_Thermal,Palingbeek_blok_maaien_2021_09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11527,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,POINT (99681.29 162242.761),99681.290204,162242.760686,4247,0.026188,0.004518,0.000374,mow,220831_Muziek_Thermal,corrected_Muziekbos_sinus_maaien_2022_06
11541,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,POINT (99728.434 162363.835),99728.434030,162363.835182,4250,0.054620,0.002679,0.000110,no_mow,220617_Muziek_fl1_Thermal,corrected_Muziekbos_blok_maaien_2021_09
11542,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,POINT (99681.29 162242.761),99681.290204,162242.760686,4251,0.026188,0.004518,0.000374,mow,220831_Muziek_Thermal,corrected_Muziekbos_sinus_maaien_2022_06
11556,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,POINT (99728.434 162363.835),99728.434030,162363.835182,4254,0.054620,0.002679,0.000110,no_mow,220617_Muziek_fl1_Thermal,corrected_Muziekbos_blok_maaien_2021_09


## Patch-first thermal variation assigned to insects

This method separates the thermal calculation from the management classification. For every thermal flight and management design (`sinus` or `blok`), it:

1. combines the mow and no-mow polygons into the full grassland;
2. divides the raster-aligned grassland into non-overlapping square patches;
3. discards patches that extend beyond the grassland or contain missing thermal pixels;
4. mean-centres each patch, calculates its DCT power spectrum, and averages power within the `<1.25`, `1.25-2.5`, and `>2.5` cycles/m bands;
5. labels a patch `edge` if it touches the 1 m-wide mow/no-mow transition zone, and otherwise labels it `mow` or `nomow`; and
6. assigns each relevant 2022 insect to exactly one patch using raster row and column indices.

`window_size_m` is the requested patch width and can be set to 2, 4, or 8 m. Because a patch must contain a whole number of raster pixels, `actual_patch_size_m` records the realized width after rounding (1.984, 3.968, or 8.000 m at a 0.064 m resolution).

The output retains every 2022 insect observation whose field, site, and survey period map to a thermal flight. Observations from unsupported fields or periods are removed. Flight-matched insects outside a valid patch retain `thermal_source` but keep their patch fields missing. `patch_x` and `patch_y` are the unrounded coordinates of the realized window centre in EPSG:31370 (metres), allowing direct Euclidean distance calculations; `patch_x_lon` and `patch_y_lat` contain the same centre in WGS84.

In [6]:
def thermal_patch_variation(thermal, shp):
    """Compute thermal power bands for one grid covering the full grassland.

    The grid is anchored to the clipped thermal raster. Only complete, finite
    windows fully covered by the combined mow/no-mow polygon are retained.
    Classification is performed after the thermal calculation; intersection
    with the 1 m-wide management boundary takes precedence over mow/no-mow.

    Returns one GeoDataFrame row per valid patch. Private grid indices are
    used to assign insects without ambiguous boundary joins; patch_x and
    patch_y contain the window centre in EPSG:31370; patch_x_lon and
    patch_y_lat contain the same centre in WGS84.
    """
    records = []
    nomow_shape = shp.geometry.iloc[0]
    mow_shape = shp.geometry.iloc[1]
    outer_shape = nomow_shape.union(mow_shape)
    edge_shape = nomow_shape.buffer(0.5).intersection(mow_shape.buffer(0.5))
    resolution_m = abs(float(thermal.rio.resolution()[0]))
    window_pixels = round(window_size_m / resolution_m)
    actual_patch_size_m = window_pixels * resolution_m
    transform = thermal.rio.transform()

    for row_start in range(0, thermal.shape[0] - window_pixels + 1, window_pixels):
        for col_start in range(0, thermal.shape[1] - window_pixels + 1, window_pixels):
            window = Window(col_start, row_start, window_pixels, window_pixels)
            west, south, east, north = window_bounds(window, transform)
            patch = box(west, south, east, north)
            values = thermal.data[
                row_start : row_start + window_pixels,
                col_start : col_start + window_pixels,
            ]
            # Exclude partial boundary windows and windows containing raster nodata.
            if not outer_shape.covers(patch) or not np.isfinite(values).all():
                continue

            # Mean-centring removes the patch-average temperature (DC component).
            psd, freqs = get_1d_psd(
                values - values.mean(),
                transform_type="DCT",
                average_power_band=True,
            )
            freqs_cpm = freqs[:-1] / actual_patch_size_m
            masks = (
                (freqs_cpm > 0) & (freqs_cpm < 1.25),
                (freqs_cpm >= 1.25) & (freqs_cpm <= 2.5),
                freqs_cpm > 2.5,
            )
            powers = [psd[mask].mean() if mask.any() else np.nan for mask in masks]

            # Classify only after computing variation; edge overlap takes precedence.
            if patch.intersects(edge_shape):
                part = "edge"
            elif mow_shape.covers(patch):
                part = "mow"
            elif nomow_shape.covers(patch):
                part = "nomow"
            else:
                continue

            x_index = col_start // window_pixels
            y_index = row_start // window_pixels
            records.append(
                {
                    "_grid_x_index": x_index,
                    "_grid_y_index": y_index,
                    "patch_x": (west + east) / 2,
                    "patch_y": (south + north) / 2,
                    "part": part,
                    "actual_patch_size_m": actual_patch_size_m,
                    **dict(
                        zip(
                            power_columns,
                            powers,
                        )
                    ),
                    "geometry": patch,
                }
            )

    patches = gpd.GeoDataFrame(records, geometry="geometry", crs=thermal.rio.crs)
    centres_wgs84 = gpd.GeoSeries(
        gpd.points_from_xy(patches["patch_x"], patches["patch_y"]), crs=patches.crs
    ).to_crs("WGS84")
    patches["patch_x_lon"] = centres_wgs84.x
    patches["patch_y_lat"] = centres_wgs84.y
    return patches

In [7]:
# Assign IDs before filtering so they remain traceable to the complete 2022 data.
patch_insects = gdf[gdf.Datum >= datetime(year=2022, month=1, day=1)].copy()
patch_insects["insect_id"] = np.arange(len(patch_insects))
patch_insects[["patch_x", "patch_y", "patch_x_lon", "patch_y_lat"]] = np.nan
patch_insects[["part", "thermal_source"]] = pd.NA
patch_insects[power_columns] = np.nan
patch_columns = [
    "patch_x",
    "patch_y",
    "patch_x_lon",
    "patch_y_lat",
    "part",
    *power_columns,
]

for flight_info, management in tqdm(thermal_shape_manifest):
    field = flight_info["field"]
    date_string = flight_info["date"]
    shp = load_shp(project_root / flight_info[f"{management}_shp"])
    outer_shape = shp.geometry.iloc[0].union(shp.geometry.iloc[1])
    therm_path = project_root / flight_info[f"{management}_tif"]
    thermal = load_thermal(
        therm_path,
        outer_shape,
        gsd_m=gsd_m,
    )
    patches = thermal_patch_variation(thermal, shp)
    assert patches.crs.to_epsg() == 31370
    window_pixels = round(window_size_m / abs(float(thermal.rio.resolution()[0])))
    # Grid indices provide a unique, constant-time patch lookup.
    patch_lookup = patches.set_index(["_grid_x_index", "_grid_y_index"])
    relevant_periods = thermal_insect_period_mapping[(field, date_string)]
    # Survey periods map observations to the thermal flight they should use.
    candidate_insects = patch_insects[
        (patch_insects["Locatie"] == field)
        & (patch_insects["Site"] == management)
        & patch_insects["Periode"].isin(relevant_periods)
    ]
    # A flight match exists independently of whether a complete patch is available.
    patch_insects.loc[candidate_insects.index, "thermal_source"] = therm_path.stem

    for insect_index, point in candidate_insects.geometry.items():
        # rowcol assigns boundary points deterministically; rows increase southward.
        row, col = rowcol(thermal.rio.transform(), point.x, point.y)
        patch_key = (col // window_pixels, row // window_pixels)
        # Missing keys correspond to discarded boundary/nodata patches.
        if patch_key not in patch_lookup.index:
            continue
        patch = patch_lookup.loc[patch_key]
        patch_insects.loc[insect_index, patch_columns] = [
            patch["patch_x"],
            patch["patch_y"],
            patch["patch_x_lon"],
            patch["patch_y_lat"],
            patch["part"],
            *patch[power_columns],
        ]

# Remove observations from fields or survey periods without a thermal flight.
patch_insects = patch_insects[patch_insects["thermal_source"].notna()].copy()

patch_insects

  0%|          | 0/16 [00:00<?, ?it/s]

,index,Datum,Periode,Locatie,Site,Transect,Soortengroep,Aantal,Genus,Lat,...,insect_id,patch_x,patch_y,patch_x_lon,patch_y_lat,part,thermal_source,average_power_lt_1p25_cpm,average_power_1p25_2p5_cpm,average_power_gt_2p5_cpm
6537,6537,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822648,...,0,47370.875196,169032.402953,2.912138,50.822672,nomow,20220617_Paling_Thermal,0.013668,0.000107,0.000003
6538,6538,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822830,...,1,NaN,NaN,NaN,NaN,<NA>,20220617_Paling_Thermal,NaN,NaN,NaN
6539,6539,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.823142,...,2,47354.875196,169088.402953,2.911895,50.823172,mow,20220617_Paling_Thermal,0.010825,0.000157,0.000006
6540,6540,2022-04-12,p1,Palingbeek,blok,T1,Dagvlinders,1,Aglais,50.822702,...,3,47402.875196,169032.402953,2.912592,50.822678,edge,20220617_Paling_Thermal,0.027464,0.000227,0.000009
6541,6541,2022-04-12,p1,Palingbeek,blok,T2,Dagvlinders,1,Aglais,50.823071,...,4,47378.875196,169080.402953,2.912238,50.823105,mow,20220617_Paling_Thermal,0.014243,0.000114,0.000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11527,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,4247,99680.356855,162239.486909,3.655422,50.768479,edge,220831_Muziek_Thermal,0.045307,0.001610,0.000087
11541,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,4250,NaN,NaN,NaN,NaN,<NA>,220617_Muziek_fl1_Thermal,NaN,NaN,NaN
11542,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,4251,99680.356855,162239.486909,3.655422,50.768479,edge,220831_Muziek_Thermal,0.045307,0.001610,0.000087
11556,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,4254,NaN,NaN,NaN,NaN,<NA>,220617_Muziek_fl1_Thermal,NaN,NaN,NaN


In [9]:
assert patch_insects["insect_id"].is_unique
assert patch_insects["thermal_source"].notna().all()
coordinate_pairs = patch_insects[["patch_x", "patch_y"]].notna()
assert coordinate_pairs["patch_x"].equals(coordinate_pairs["patch_y"])
patch_matched = coordinate_pairs["patch_x"]
assert patch_insects.loc[patch_matched, "part"].isin(["mow", "nomow", "edge"]).all()
power_values = patch_insects[power_columns].to_numpy()
complete_triplets = np.isfinite(power_values).all(axis=1) | np.isnan(power_values).all(axis=1)
assert complete_triplets.all(), "Power bands must be either all finite or all NaN"

patch_insect_output_path = output_root / (
    f"insects_with_mapped_patch_thermal_variation_v2_{window_size_m:g}x{window_size_m:g}m_"
    f"{gsd_m:g}m_gsd_1p25-2p5_cycles_per_m.xlsx"
)
patch_insects.drop(columns="geometry").to_excel(patch_insect_output_path, index=False)
print(
    f"Wrote {len(patch_insects)} flight-matched insects "
    f"({patch_matched.sum()} with valid patches) to "
    f"{patch_insect_output_path}"
)
patch_insects.groupby(["Locatie", "Site"], as_index=False).agg(
    insects=("insect_id", "size"), insects_with_patches=("patch_x", "count")
)

Wrote 2330 flight-matched insects (1295 with valid patches) to /home/emield/docs/PhD/projects/thermal-image-analysis/github/thermal-microclimate/data/derived/insect_analysis/insects_with_mapped_patch_thermal_variation_v2_8x8m_0.064m_gsd_1p25-2p5_cycles_per_m.xlsx


,Locatie,Site,insects,insects_with_patches
0,Muziekbos,blok,370,232
1,Muziekbos,sinus,507,184
2,Palingbeek,blok,382,295
3,Palingbeek,sinus,554,316
4,Waarmaarde,blok,220,133
5,Waarmaarde,sinus,297,135


In [17]:
patch_insects

,index,Datum,Periode,Locatie,Site,Transect,Soortengroep,Aantal,Genus,Lat,...,insect_id,patch_x,patch_y,patch_x_lon,patch_y_lat,part,thermal_source,average_power_lt_1p25_cpm,average_power_1p25_2p5_cpm,average_power_gt_2p5_cpm
6537,6537,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822648,...,0,47367.227196,169030.098953,2.912087,50.822651,nomow,20220617_Paling_Thermal,0.013484,0.002192,0.000052
6538,6538,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.822830,...,1,47301.755196,169051.922953,2.911151,50.822835,mow,20220617_Paling_Thermal,0.083190,0.004213,0.000120
6539,6539,2022-04-12,p1,Palingbeek,blok,V,Dagvlinders,1,Aglais,50.823142,...,2,47357.307196,169085.650953,2.911930,50.823148,mow,20220617_Paling_Thermal,0.020885,0.001424,0.000062
6540,6540,2022-04-12,p1,Palingbeek,blok,T1,Dagvlinders,1,Aglais,50.822702,...,3,47402.939196,169036.050953,2.912592,50.822710,nomow,20220617_Paling_Thermal,0.161257,0.008797,0.000137
6541,6541,2022-04-12,p1,Palingbeek,blok,T2,Dagvlinders,1,Aglais,50.823071,...,4,47375.163196,169075.730953,2.912186,50.823062,mow,20220617_Paling_Thermal,0.065113,0.003778,0.000106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11527,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,4247,99681.060855,162242.750909,3.655431,50.768508,mow,220831_Muziek_Thermal,0.053257,0.016916,0.001523
11541,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,4250,99727.892131,162364.155566,3.656078,50.769603,nomow,220617_Muziek_fl1_Thermal,0.053874,0.008745,0.000400
11542,10336,2022-08-31,p5,Muziekbos,sinus,V,Dagvlinders,1,Polyommatus,50.768508,...,4251,99681.060855,162242.750909,3.655431,50.768508,mow,220831_Muziek_Thermal,0.053257,0.016916,0.001523
11556,9349,2022-06-20,p3,Muziekbos,blok,V,Dagvlinders,1,Maniola,50.769601,...,4254,99727.892131,162364.155566,3.656078,50.769603,nomow,220617_Muziek_fl1_Thermal,0.053874,0.008745,0.000400


In [15]:
# Count Nan and group them by periode, Date and transect
patch_insects[patch_insects.Locatie=="Muziekbos"].groupby(["Locatie", "Site", "Periode", "Datum"], as_index=False).agg(
    insects=("insect_id", "size"),
    insects_with_patches=("patch_x", "count"),
)
# patch_insects[(patch_insects.Locatie=="Waarmaarde") & (patch_insects.Site=="sinus")].

,Locatie,Site,Periode,Datum,insects,insects_with_patches
0,Muziekbos,blok,p1,2022-04-15,28,17
1,Muziekbos,blok,p2,2022-05-14,51,28
2,Muziekbos,blok,p3,2022-06-20,97,66
3,Muziekbos,blok,p4,2022-08-06,87,50
4,Muziekbos,blok,p5,2022-08-31,65,51
5,Muziekbos,blok,p6,2022-09-21,42,20
6,Muziekbos,sinus,p1,2022-04-15,36,15
7,Muziekbos,sinus,p1,2022-05-14,43,23
8,Muziekbos,sinus,p2,2022-05-14,28,12
9,Muziekbos,sinus,p3,2022-06-20,134,32


In [16]:
patch_insects.columns

Index(['index', 'Datum', 'Periode', 'Locatie', 'Site', 'Transect',
       'Soortengroep', 'Aantal', 'Genus', 'Nauwkeurigheid', 'Meettoestel',
       'Jaar', 'geometry', 'x', 'y', 'insect_id', 'patch_x', 'patch_y', 'part',
       'thermal_source', 'average_power_lt_1p25_cpm',
       'average_power_1p25_2p5_cpm', 'average_power_gt_2p5_cpm'],
      dtype='str')